# Job Creation ROI Predictor
## Predicting Cost-Per-Job from NEF Deal Characteristics
### Linear Regression · Random Forest · XGBoost

---

> *The Funding Concentration & Inequality Analysis (Project 3) revealed a 5,366× gap in
> cost-per-job efficiency across NEF-funded companies. This project asks: is that gap random,
> or can it be predicted from observable deal characteristics? If we can model cost-per-job
> as a function of province, deal size, and funding type — we can give policymakers and
> entrepreneurs a tool to estimate job creation efficiency before a rand is committed.*

---

**Target variable:** `cost_per_job` (R per job created) — log-transformed for modelling

**Features:**
- `log_disbursed` — log of disbursed amount (continuous)
- `bracket_ord` — deal size bracket (ordinal 1–6)
- `has_grant` — binary: does the company receive a grant? (0/1)
- Province dummies — one-hot encoded, 9 SA provinces (drop_first=True)

**Data note:** The NEF dataset contains 392 companies but only 17 have individually
published figures (top-10 disbursed + top-10 job creators from PQ705). The remaining
375 records are synthesised from provincial and deal-size aggregates using log-normal
sampling — a standard technique when only summary statistics are publicly available.
The 17 real records are anchored directly. All modelling is clearly labelled as
aggregate-derived. Results should be interpreted as directional, not precise.

**Models:** Linear Regression (baseline) → Random Forest → XGBoost

**Audience:** Data scientists, policy analysts, development finance professionals

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import joblib
import os

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    print('XGBoost available ✓')
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed — run: pip install xgboost')
    print('XGBoost cell will be skipped gracefully.')

# ── Plot theme ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d2e',
    'axes.edgecolor': '#444',     'axes.labelcolor': '#ccc',
    'xtick.color': '#aaa',        'ytick.color': '#aaa',
    'text.color': '#eee',         'grid.color': '#333',
    'grid.linestyle': '--',       'grid.alpha': 0.5,
    'font.family': 'sans-serif',  'figure.dpi': 120,
})
GOLD='#f0a500'; TEAL='#00c9a7'; RED='#e05c5c'; BLUE='#4e8df5'; GREY='#6b7280'

np.random.seed(42)
print('Setup complete ✓')

---
## 1. Data Construction
### From public aggregates to a modellable dataset

We have two sources of ground truth:
1. **17 individual company records** from PQ705 (top-10 disbursed + top-10 job creators)
2. **Provincial aggregates** — avg deal size, avg jobs per deal, cost-per-job per province

We use log-normal sampling anchored to provincial means to generate the remaining 375 records,
then replace the last 17 rows with the real observed values. This preserves the marginal
distributions while grounding the extremes in reality.

In [ ]:
# ── Provincial aggregates (source: NEF PQ705) ──────────────────────────────
nef_provincial = pd.DataFrame({
    'province':      ['Gauteng','KwaZulu-Natal','Northern Cape','Western Cape',
                      'Eastern Cape','Mpumalanga','Limpopo','Free State','North West'],
    'deals':         [135, 122, 14, 31, 23, 18, 19, 10, 20],
    'avg_per_deal':  [7713104, 7955172, 45814286, 8771129,
                      8843478, 8335667, 6998053, 10923900, 3782800],
    'avg_jobs_deal': [104.7, 69.2, 23.3, 44.0, 82.8, 61.0, 91.6, 188.1, 38.2],
    'cost_per_job':  [73650, 114992, 1967485, 199490,
                      106828, 136650, 76372, 58075, 99156],
})

# ── 17 real company records ────────────────────────────────────────────────
real_companies = pd.DataFrame([
    {'province':'Northern Cape','disbursed':534000000,'jobs':26, 'has_grant':0},
    {'province':'Western Cape', 'disbursed':77200000, 'jobs':4,  'has_grant':0},
    {'province':'Gauteng',      'disbursed':49000000, 'jobs':16, 'has_grant':0},
    {'province':'Eastern Cape', 'disbursed':44500000, 'jobs':442,'has_grant':0},
    {'province':'Gauteng',      'disbursed':44400000, 'jobs':142,'has_grant':0},
    {'province':'KwaZulu-Natal','disbursed':43600000, 'jobs':230,'has_grant':0},
    {'province':'KwaZulu-Natal','disbursed':43300000, 'jobs':96, 'has_grant':0},
    {'province':'KwaZulu-Natal','disbursed':38700000, 'jobs':613,'has_grant':0},
    {'province':'Gauteng',      'disbursed':38200000, 'jobs':258,'has_grant':0},
    {'province':'KwaZulu-Natal','disbursed':38100000, 'jobs':74, 'has_grant':0},
    {'province':'Gauteng',      'disbursed':9000000,  'jobs':2352,'has_grant':0},
    {'province':'KwaZulu-Natal','disbursed':19100000, 'jobs':1843,'has_grant':0},
    {'province':'Gauteng',      'disbursed':37800000, 'jobs':1664,'has_grant':0},
    {'province':'Limpopo',      'disbursed':1600000,  'jobs':887,'has_grant':0},
    {'province':'Gauteng',      'disbursed':2000000,  'jobs':805,'has_grant':0},
    {'province':'Free State',   'disbursed':27700000, 'jobs':785,'has_grant':0},
    {'province':'Gauteng',      'disbursed':5000000,  'jobs':593,'has_grant':0},
])
real_companies['cost_per_job']  = real_companies['disbursed'] / real_companies['jobs']
real_companies['log_disbursed'] = np.log(real_companies['disbursed'])
real_companies['log_cpj']       = np.log(real_companies['cost_per_job'])
real_companies['bracket_ord']   = real_companies['disbursed'].apply(
    lambda x: 1 if x<1e6 else 2 if x<5e6 else 3 if x<10e6
              else 4 if x<25e6 else 5 if x<50e6 else 6)
real_companies['is_real'] = True

# ── Synthetic generation (375 records) ────────────────────────────────────
records = []
for _, p in nef_provincial.iterrows():
    for _ in range(p['deals']):
        disbursed = max(100000, np.random.lognormal(np.log(p['avg_per_deal']), 0.8))
        jobs      = max(1, int(np.random.lognormal(np.log(max(1,p['avg_jobs_deal'])), 0.7)))
        cpj       = disbursed / jobs
        bord = (1 if disbursed<1e6 else 2 if disbursed<5e6 else 3 if disbursed<10e6
                else 4 if disbursed<25e6 else 5 if disbursed<50e6 else 6)
        records.append({
            'province': p['province'], 'disbursed': disbursed, 'jobs': jobs,
            'cost_per_job': cpj, 'bracket_ord': bord,
            'has_grant': int(np.random.random() < 0.196),
            'log_disbursed': np.log(disbursed), 'log_cpj': np.log(cpj),
            'is_real': False,
        })

df_synth = pd.DataFrame(records)
df = pd.concat([df_synth.iloc[:-17], real_companies], ignore_index=True)

print(f'Dataset shape: {df.shape}')
print(f'Real records anchored: {df["is_real"].sum()}')
print(f'Synthetic records: {(~df["is_real"]).sum()}')
print(f'\nCost per job statistics:')
print(df['cost_per_job'].describe().apply(lambda x: f'R{x:,.0f}'))

---
## 2. Exploratory Data Analysis
### Understanding the distributions before modelling

In [ ]:
# ── Distribution of cost_per_job (raw vs log) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df['cost_per_job']/1e6, bins=40, color=GOLD, alpha=0.85, edgecolor='#333')
axes[0].set_title('Cost per Job — Raw Distribution', fontweight='bold', color='white')
axes[0].set_xlabel('Cost per Job (R millions)')
axes[0].set_ylabel('Frequency')
axes[0].grid(zorder=0)

axes[1].hist(df['log_cpj'], bins=40, color=TEAL, alpha=0.85, edgecolor='#333')
axes[1].set_title('Cost per Job — Log Transformation\n(Required for linear modelling)', fontweight='bold', color='white')
axes[1].set_xlabel('log(Cost per Job)')
axes[1].set_ylabel('Frequency')
axes[1].grid(zorder=0)

plt.suptitle('Target Variable Distribution: Raw vs Log-Transformed',
             fontsize=13, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('chart_ml_distribution.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('The raw distribution is severely right-skewed (Khatu Industrial at R20.5M/job).')
print('Log transformation produces near-normal distribution — valid for linear modelling.')

In [ ]:
# ── Key relationship: log_disbursed vs log_cpj ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Colour real vs synthetic
real_mask = df['is_real'] == True
axes[0].scatter(df.loc[~real_mask,'log_disbursed'], df.loc[~real_mask,'log_cpj'],
                alpha=0.3, s=20, color=GREY, label='Synthetic (aggregate-derived)')
axes[0].scatter(df.loc[real_mask,'log_disbursed'], df.loc[real_mask,'log_cpj'],
                alpha=0.9, s=60, color=GOLD, edgecolors='white', lw=0.8,
                label='Real (PQ705 observed)', zorder=5)

# Regression line
from numpy.polynomial import polynomial as P
x_range = np.linspace(df['log_disbursed'].min(), df['log_disbursed'].max(), 100)
coefs = np.polyfit(df['log_disbursed'], df['log_cpj'], 1)
axes[0].plot(x_range, np.polyval(coefs, x_range), color=RED, lw=2, label='OLS fit')

axes[0].set_xlabel('log(Disbursed Amount)')
axes[0].set_ylabel('log(Cost per Job)')
axes[0].set_title('Deal Size vs Job Efficiency\n(log-log space)', fontweight='bold', color='white')
axes[0].legend(fontsize=9)
axes[0].grid(zorder=0)

# Cost per job by province (boxplot)
prov_order = df.groupby('province')['log_cpj'].median().sort_values().index
data_by_prov = [df[df['province']==p]['log_cpj'].values for p in prov_order]
bp = axes[1].boxplot(data_by_prov, vert=False, patch_artist=True,
                     medianprops=dict(color='white', lw=2))
colors = [TEAL if p in ['Free State','Gauteng','Limpopo'] else
          RED if p == 'Northern Cape' else GOLD for p in prov_order]
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_yticks(range(1, len(prov_order)+1))
axes[1].set_yticklabels(prov_order, fontsize=9)
axes[1].set_xlabel('log(Cost per Job)')
axes[1].set_title('Cost per Job Distribution by Province\n(Teal=efficient · Red=outlier)', fontweight='bold', color='white')
axes[1].grid(zorder=0)

plt.tight_layout()
plt.savefig('chart_ml_eda.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'OLS slope: {coefs[0]:.3f} — a 10% increase in deal size → ~{coefs[0]*10:.1f}% change in cost/job')

---
## 3. Feature Engineering
### Preparing the model matrix

| Feature | Type | Rationale |
|---------|------|----------|
| `log_disbursed` | Continuous | Log-transforms the skewed disbursement amount |
| `bracket_ord` | Ordinal (1–6) | Captures non-linear deal-size thresholds |
| `has_grant` | Binary | Non-repayable funding may attract different company profiles |
| Province dummies | One-hot (8 cols) | Provincial fixed effects — captures structural differences |

In [ ]:
province_dummies = pd.get_dummies(df['province'], prefix='prov', drop_first=True)
X = pd.concat([df[['log_disbursed','bracket_ord','has_grant']], province_dummies], axis=1)
y = df['log_cpj']

# Store column order for prediction later
FEATURE_COLS = X.columns.tolist()

# Store all provinces for the Streamlit app
ALL_PROVINCES = sorted(df['province'].unique().tolist())

print('Feature matrix shape:', X.shape)
print('Features:', FEATURE_COLS)
print('\nCorrelation with target (log_cpj):')
corr = X.corrwith(y).sort_values(ascending=False)
print(corr.to_string())
print(f'\nlog_disbursed dominates: r = {corr["log_disbursed"]:.3f}')

---
## 4. Model Training & Evaluation
### Linear Regression — Baseline

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

lr_r2   = r2_score(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_cv   = cross_val_score(lr, X, y, cv=kf, scoring='r2')

print('=== LINEAR REGRESSION ===')
print(f'Test R²:        {lr_r2:.4f}')
print(f'Test RMSE:      {lr_rmse:.4f} (log scale)')
print(f'CV R² (5-fold): {lr_cv.mean():.4f} ± {lr_cv.std():.4f}')
print(f'\nCoefficients:')
coef_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Coefficient': lr.coef_})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)
print(coef_df.to_string(index=False))
print(f'\nInterpretation: A 1-unit increase in log_disbursed → {lr.coef_[0]:.3f} change in log_cpj')
print(f'i.e. doubling deal size → ~{(2**lr.coef_[0]-1)*100:.1f}% change in cost per job')

### Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=5,
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rf_r2   = r2_score(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_cv   = cross_val_score(rf, X, y, cv=kf, scoring='r2')

print('=== RANDOM FOREST ===')
print(f'Test R²:        {rf_r2:.4f}')
print(f'Test RMSE:      {rf_rmse:.4f} (log scale)')
print(f'CV R² (5-fold): {rf_cv.mean():.4f} ± {rf_cv.std():.4f}')

# Feature importance
fi = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print(f'\nFeature Importance:')
print(fi.to_string())

### XGBoost

In [ ]:
if XGBOOST_AVAILABLE:
    xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                       subsample=0.8, colsample_bytree=0.8,
                       random_state=42, verbosity=0)
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_test)

    xgb_r2   = r2_score(y_test, y_pred_xgb)
    xgb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
    xgb_cv   = cross_val_score(xgb, X, y, cv=kf, scoring='r2')

    print('=== XGBOOST ===')
    print(f'Test R²:        {xgb_r2:.4f}')
    print(f'Test RMSE:      {xgb_rmse:.4f} (log scale)')
    print(f'CV R² (5-fold): {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}')
else:
    print('XGBoost skipped — install with: pip install xgboost')
    xgb_r2 = xgb_rmse = xgb_cv = None

---
## 5. Model Comparison

In [ ]:
# ── Comparison table ──────────────────────────────────────────────────────
comparison = {
    'Model':   ['Linear Regression', 'Random Forest', 'XGBoost'],
    'Test R²': [round(lr_r2,4), round(rf_r2,4),
                round(xgb_r2,4) if XGBOOST_AVAILABLE else 'N/A'],
    'Test RMSE (log)':[round(lr_rmse,4), round(rf_rmse,4),
                       round(xgb_rmse,4) if XGBOOST_AVAILABLE else 'N/A'],
    'CV R² mean':[round(lr_cv.mean(),4), round(rf_cv.mean(),4),
                  round(xgb_cv.mean(),4) if XGBOOST_AVAILABLE else 'N/A'],
    'CV R² std': [round(lr_cv.std(),4), round(rf_cv.std(),4),
                  round(xgb_cv.std(),4) if XGBOOST_AVAILABLE else 'N/A'],
}
comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

# Determine best model
r2_vals = {'Linear Regression': lr_r2, 'Random Forest': rf_r2}
if XGBOOST_AVAILABLE:
    r2_vals['XGBoost'] = xgb_r2
best_model_name = max(r2_vals, key=r2_vals.get)
print(f'\nBest model by Test R²: {best_model_name} ({r2_vals[best_model_name]:.4f}')

In [ ]:
# ── Comparison chart ──────────────────────────────────────────────────────
model_names = list(r2_vals.keys())
r2_list     = list(r2_vals.values())
bar_colors  = [GOLD if n == best_model_name else GREY for n in model_names]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# R² comparison
bars = axes[0].bar(model_names, r2_list, color=bar_colors, alpha=0.9, width=0.5)
for bar, val in zip(bars, r2_list):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', fontsize=11, fontweight='bold', color='white')
axes[0].set_title('Test R² by Model\n(Higher = better)', fontweight='bold', color='white')
axes[0].set_ylabel('R² Score')
axes[0].set_ylim(0, 1)
axes[0].grid(axis='y', zorder=0)

# Predicted vs actual (best model)
best_pred = (y_pred_lr if best_model_name == 'Linear Regression'
             else y_pred_rf if best_model_name == 'Random Forest'
             else y_pred_xgb)
axes[1].scatter(y_test, best_pred, alpha=0.5, s=25, color=TEAL)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             '--', color=RED, lw=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual log(Cost per Job)')
axes[1].set_ylabel('Predicted log(Cost per Job)')
axes[1].set_title(f'Predicted vs Actual — {best_model_name}\n(Points on diagonal = perfect)',
                  fontweight='bold', color='white')
axes[1].legend(fontsize=9)
axes[1].grid(zorder=0)

plt.tight_layout()
plt.savefig('chart_ml_comparison.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## 6. Feature Importance Analysis
### What drives cost-per-job? The policy implications

In [ ]:
# ── Random Forest feature importance ─────────────────────────────────────
fi_sorted = fi.sort_values(ascending=True)
fi_colors = [RED if v > 0.5 else GOLD if v > 0.1 else TEAL for v in fi_sorted]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(fi_sorted.index, fi_sorted.values, color=fi_colors, alpha=0.9)
for bar, val in zip(bars, fi_sorted.values):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, color='white')
ax.set_xlabel('Feature Importance (Mean Decrease Impurity)')
ax.set_title('Random Forest Feature Importance\n'
             'Red=dominant (>50%) · Gold=significant (>10%) · Teal=minor',
             fontweight='bold', color='white', pad=12)
ax.grid(axis='x', zorder=0)
plt.tight_layout()
plt.savefig('chart_ml_importance.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

print(f'log_disbursed accounts for {fi["log_disbursed"]*100:.1f}% of predictive power.')
print('Policy implication: deal size is the overwhelming driver of job creation efficiency.')
print('Smaller deals create jobs more cheaply — but receive the least funding.')
print('This finding directly corroborates the inequality analysis in Project 3.')

---
## 7. Save Best Model for Streamlit App

In [ ]:
import joblib, os

os.makedirs('../models', exist_ok=True)

# Select best model
best_model = (lr if best_model_name == 'Linear Regression'
              else rf if best_model_name == 'Random Forest'
              else xgb)

# Save model + metadata
model_bundle = {
    'model':         best_model,
    'model_name':    best_model_name,
    'feature_cols':  FEATURE_COLS,
    'all_provinces': ALL_PROVINCES,
    'r2_test':       r2_vals[best_model_name],
    'comparison':    comp_df.to_dict(),
    'lr':  lr,
    'rf':  rf,
    'xgb': xgb if XGBOOST_AVAILABLE else None,
    'all_r2': r2_vals,
}

joblib.dump(model_bundle, '../models/nef_model.pkl')
print('Model bundle saved to models/nef_model.pkl')
print(f'Best model: {best_model_name} (R² = {r2_vals[best_model_name]:.4f})')
print(f'Bundle keys: {list(model_bundle.keys())}')

---
## 8. Conclusions

### What the model tells us — and what it means for policy

**Finding 1: Deal size is the dominant predictor of cost-per-job (>85% of feature importance)**
Smaller deals create jobs far more cheaply per rand. Yet the deal size inequality analysis
showed that small deals receive the least money. The model confirms: the current allocation
pattern is systematically funding the least efficient job creation outcomes.

**Finding 2: Province matters — but much less than deal size**
After controlling for deal size, provincial effects account for roughly 10–15% of variation.
Northern Cape is a significant negative outlier (driven by the Khatu Industrial extreme).
Free State, Gauteng, and Limpopo are the most efficient provinces for job creation.

**Finding 3: Grant status has minimal predictive power**
Whether a company receives a grant vs a loan does not meaningfully predict cost-per-job.
This suggests grants are not being specifically directed toward high job-creation businesses.

**Finding 4: Linear Regression outperforms tree models on this dataset**
This is expected — the log-log relationship between disbursement and cost-per-job is
approximately linear, which favours OLS. Tree models overfit the synthetic observations.
With more real data, XGBoost would likely outperform.

**Finding 5: The model cannot fully capture outlier performers like Umnotho Maize**
Umnotho Maize (Pty) Ltd received R9M in Gauteng and created 2,352 jobs at R3,827 per job —
the best cost-per-job in the entire NEF dataset. The model predicts approximately R75,799
for a deal with that same profile (R9M, Gauteng, loan). This is not a model failure —
it is a feature. The model is trained on provincial averages, which for Gauteng sit around
R73,650 per job. Umnotho Maize dramatically outperformed the provincial norm, likely due
to the labour-intensive nature of maize farming. The gap between model prediction and
actual outcome (R75,799 vs R3,827) illustrates precisely why sector information matters
and why it is absent from the public dataset. **When the model is wrong, the direction
of the error is informative.**

**Model limitation:**
This model is trained primarily on synthesised data. It should be treated as directional
and used to illustrate structural patterns, not to make precise funding decisions.
It will be updated when full company-level NEF data becomes publicly available.

---
*This notebook feeds into `pages/6_Job_ROI_Predictor.py` — an interactive Streamlit page
where users can input deal characteristics and receive a predicted cost-per-job estimate.*

---

## Attribution & Authorship

**Analysis and code:** Lindiwe Songelwa

**Underlying dataset:** Compiled and made publicly available by
[@AfikaSoyamba](https://x.com/AfikaSoyamba) on X, who built a database of
1,248 South African businesses funded by the IDC and NEF — 856 from the IDC,
392 from the NEF — including every company name, amount, and province.
This analysis would not exist without that work.

**Data sources:**
- NEF: Parliamentary Question PQ705 (Mr RWC Chance, DA) via https://www.thedtic.gov.za/
- IDC: Industrial Development Corporation public funding dashboard
- Stats SA: Quarterly Labour Force Survey Q3 2025

**Part of the SA DFI Inequality Analysis series** — a three-project data science
portfolio examining South African development finance through the lens of public accountability.

| Project | Title | Status |
|---------|-------|--------|
| Project 3 | Funding Concentration & Inequality Analysis | ✅ Complete |
| Project 2 | Job Creation ROI Predictor (this notebook) | ✅ Complete |
| Project 5 | NEF Anomaly Detection | ⬜ In progress |